# Aya-Expanse-8B: Multilingual LoRA Fine-Tuning (WMT26 Unconstrained Track)
Trains a single multilingual LoRA adapter across:
- English → Chinese (Simplified)
- English → Arabic
- Czech → German

Continues from a previously trained LoRA adapter (en-zh).

In [ ]:
# pip dependency warnings about numba/dask-cuda on Kaggle are harmless — ignore them.
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub sentencepiece

In [ ]:
import os

OUTPUT_DIR   = "/kaggle/working/aya-multilingual-lora"
FINAL_DIR    = "/kaggle/working/aya-multilingual-lora-final"
PREV_LORA    = "/kaggle/input/datasets/anishracherla06/arya-english-chineese"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR,  exist_ok=True)

print(f"Output dir       : {OUTPUT_DIR}")
print(f"Previous LoRA    : {PREV_LORA}")
print(f"Prev LoRA exists : {os.path.exists(PREV_LORA)}")

In [ ]:
from huggingface_hub import login
hf_token = input("Enter your Hugging Face WRITE token: ")
login(token=hf_token)

## Dataset Loading & Formatting (OPUS-100)

In [ ]:
from datasets import load_dataset, concatenate_datasets

MAX_PER_LANG = 50_000

# ------------------------------------------------------------------
# Helper: load an OPUS-100 split safely
# ------------------------------------------------------------------
def load_opus(config, split="train"):
    """Load OPUS-100 dataset; fall back to full split if train missing."""
    try:
        ds = load_dataset("opus100", config, split=split)
    except Exception:
        ds = load_dataset("opus100", config, split="train")
    return ds

# ------------------------------------------------------------------
# 1. English → Chinese (Simplified)
# ------------------------------------------------------------------
print("[1/3] Loading OPUS-100 en-zh ...")
ds_enzh_raw = load_opus("en-zh")
print(f"      Raw: {len(ds_enzh_raw):,}")

def fmt_enzh(ex):
    src = ex["translation"]["en"]
    tgt = ex["translation"]["zh"]
    return {"text": (
        "Source language: English\n"
        "Target language: Chinese\n"
        "Translate the following text:\n"
        + str(src) + "\n\nResponse:\n" + str(tgt)
    )}

ds_enzh = (
    ds_enzh_raw
    .shuffle(seed=42)
    .select(range(min(MAX_PER_LANG, len(ds_enzh_raw))))
    .map(fmt_enzh, remove_columns=ds_enzh_raw.column_names)
)
print(f"      Sampled: {len(ds_enzh):,}")

# ------------------------------------------------------------------
# 2. English → Arabic
# ------------------------------------------------------------------
print("[2/3] Loading OPUS-100 en-ar ...")
ds_enar_raw = load_opus("en-ar")
print(f"      Raw: {len(ds_enar_raw):,}")

def fmt_enar(ex):
    src = ex["translation"]["en"]
    tgt = ex["translation"]["ar"]
    return {"text": (
        "Source language: English\n"
        "Target language: Arabic\n"
        "Translate the following text:\n"
        + str(src) + "\n\nResponse:\n" + str(tgt)
    )}

ds_enar = (
    ds_enar_raw
    .shuffle(seed=42)
    .select(range(min(MAX_PER_LANG, len(ds_enar_raw))))
    .map(fmt_enar, remove_columns=ds_enar_raw.column_names)
)
print(f"      Sampled: {len(ds_enar):,}")

# ------------------------------------------------------------------
# 3. Czech → German
# ------------------------------------------------------------------
print("[3/3] Loading OPUS-100 cs-de ...")
ds_csde_raw = load_opus("cs-de")
print(f"      Raw: {len(ds_csde_raw):,}")

def fmt_csde(ex):
    src = ex["translation"]["cs"]
    tgt = ex["translation"]["de"]
    return {"text": (
        "Source language: Czech\n"
        "Target language: German\n"
        "Translate the following text:\n"
        + str(src) + "\n\nResponse:\n" + str(tgt)
    )}

ds_csde = (
    ds_csde_raw
    .shuffle(seed=42)
    .select(range(min(MAX_PER_LANG, len(ds_csde_raw))))
    .map(fmt_csde, remove_columns=ds_csde_raw.column_names)
)
print(f"      Sampled: {len(ds_csde):,}")

In [ ]:
# ------------------------------------------------------------------
# Verification: counts and sample prompts
# ------------------------------------------------------------------
print("=" * 55)
print(f"  English → Chinese  : {len(ds_enzh):>7,} examples")
print(f"  English → Arabic   : {len(ds_enar):>7,} examples")
print(f"  Czech   → German   : {len(ds_csde):>7,} examples")

combined_all = concatenate_datasets([ds_enzh, ds_enar, ds_csde]).shuffle(seed=42)
print(f"  Combined total     : {len(combined_all):>7,} examples")
print("=" * 55)

print("\n--- Sample: English → Chinese ---")
print(ds_enzh[0]["text"][:300])
print("\n--- Sample: English → Arabic ---")
print(ds_enar[0]["text"][:300])
print("\n--- Sample: Czech → German ---")
print(ds_csde[0]["text"][:300])

In [ ]:
# Train / eval split from combined dataset
splits = combined_all.train_test_split(test_size=1000, seed=42)
train_dataset = splits["train"]
eval_dataset  = splits["test"]
print(f"Train: {len(train_dataset):,}  |  Eval: {len(eval_dataset):,}")

## Load Aya 8B Base Model (4-bit)

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

MODEL_ID = "CohereForAI/aya-expanse-8b"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

torch.cuda.empty_cache()
gc.collect()

print("Loading Aya-Expanse-8B in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.torch_dtype = torch.float16
model = prepare_model_for_kbit_training(model)
print("✅ Base model loaded.")

## Load Previous LoRA Adapter & Continue Training

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

adapter_cfg = os.path.join(PREV_LORA, "adapter_config.json")
adapter_wts = os.path.join(PREV_LORA, "adapter_model.safetensors")

if os.path.isfile(adapter_cfg):
    print(f"🔄 Loading previous LoRA from: {PREV_LORA}")
    model = PeftModel.from_pretrained(model, PREV_LORA, is_trainable=True)
    print(f"   adapter_config.json      : {'✅' if os.path.isfile(adapter_cfg) else '❌'}")
    print(f"   adapter_model.safetensors: {'✅' if os.path.isfile(adapter_wts) else '❌'}")
    print("✅ Previous LoRA loaded and set to trainable.")
else:
    print(f"⚠️  No adapter_config.json at {PREV_LORA}. Starting with fresh LoRA.")
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

# Cast LoRA params to float32 — required to avoid bfloat16 grad bug on T4
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

print("✅ LoRA adapter ready.")

## Training

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers.trainer_utils import get_last_checkpoint

torch.cuda.empty_cache()

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    learning_rate=1e-4,
    fp16=False,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    push_to_hub=False,
    report_to="none",
    dataset_text_field="text",
    packing=False,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

# Auto-resume logic
last_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    last_ckpt = get_last_checkpoint(OUTPUT_DIR)

print("\n" + "="*55)
if last_ckpt:
    print(f"🚀 Resuming from checkpoint: {last_ckpt}")
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("🌟 Starting fresh continued training.")
    trainer.train()

# Save final adapter
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"\n🎉 Training complete! Adapter saved to: {FINAL_DIR}")

## Evaluation — FLORES-200 (COMET)

In [ ]:
# Install COMET after training (requires older transformers — safe now)
!pip install -q evaluate unbabel-comet pytorch-lightning "transformers<4.45"

In [ ]:
import torch, tarfile, urllib.request, tempfile, os, json
from tqdm.auto import tqdm

FLORES_URL     = "https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz"
SAMPLES        = 200

EVAL_PAIRS = [
    {
        "name": "English → Chinese",
        "src": "./flores200_dataset/dev/eng_Latn.dev",
        "ref": "./flores200_dataset/dev/zho_Hans.dev",
        "prompt": "Source language: English\nTarget language: Chinese\nTranslate the following text:\n{src}\n\nResponse:\n",
        "json": "flores_enzh.json",
    },
    {
        "name": "English → Arabic",
        "src": "./flores200_dataset/dev/eng_Latn.dev",
        "ref": "./flores200_dataset/dev/arb_Arab.dev",
        "prompt": "Source language: English\nTarget language: Arabic\nTranslate the following text:\n{src}\n\nResponse:\n",
        "json": "flores_enar.json",
    },
    {
        "name": "Czech → German",
        "src": "./flores200_dataset/dev/ces_Latn.dev",
        "ref": "./flores200_dataset/dev/deu_Latn.dev",
        "prompt": "Source language: Czech\nTarget language: German\nTranslate the following text:\n{src}\n\nResponse:\n",
        "json": "flores_csde.json",
    },
]

print("Downloading FLORES-200...")
with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
    urllib.request.urlretrieve(FLORES_URL, tmp.name)
    tar_path = tmp.name
print("Done.")

model.eval()

for pair in EVAL_PAIRS:
    print(f"\n{'='*50}\n{pair['name']}\n{'='*50}")
    with tarfile.open(tar_path, "r:gz") as tar:
        srcs = [l.decode().strip() for l in tar.extractfile(pair["src"]).readlines()][:SAMPLES]
        refs = [l.decode().strip() for l in tar.extractfile(pair["ref"]).readlines()][:SAMPLES]

    preds = []
    for s in tqdm(srcs, desc=pair["name"]):
        inp = tokenizer(pair["prompt"].format(src=s), return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=200,
                                 pad_token_id=tokenizer.eos_token_id, do_sample=False)
        n = out.shape[1] - inp.input_ids.shape[1]
        preds.append(tokenizer.decode(out[0][-n:], skip_special_tokens=True).strip().replace("\n"," "))

    data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(srcs, preds, refs)]
    with open(pair["json"], "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    print(f"Sample: {preds[0][:120]}")

os.remove(tar_path)
print("\n✅ All translations generated.")

In [ ]:
import json

pairs_to_score = [
    ("flores_enzh.json", "English → Chinese"),
    ("flores_enar.json", "English → Arabic"),
    ("flores_csde.json", "Czech → German"),
]

for jf, name in pairs_to_score:
    script = f"""
import json, logging
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
from comet import download_model, load_from_checkpoint
model_path = download_model("Unbabel/wmt22-comet-da")
cm = load_from_checkpoint(model_path)
with open("{jf}", "r", encoding="utf-8") as f:
    data = json.load(f)
res = cm.predict(data, batch_size=8, gpus=1)
print("=" * 50)
print(f"COMET [{name}] ({{len(data)}} samples): {{res.system_score:.4f}}")
print("=" * 50)
"""
    sname = f"comet_{jf.replace('.json','')}.py"
    with open(sname, "w") as f:
        f.write(script)
    print(f"\nScoring {name}...")
    !python {sname}

In [ ]:
import shutil
from IPython.display import FileLink

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(FINAL_DIR) for f in files
)
print(f"Adapter size: {total / 1024**2:.2f} MB")

zip_out = "/kaggle/working/aya-multilingual-lora-final"
shutil.make_archive(zip_out, "zip", FINAL_DIR)
print("\n✅ Download your adapter:")
display(FileLink("aya-multilingual-lora-final.zip"))